Reference from [Introduction to Spherical Harmonics for Graphics Programmers](https://gpfault.net/posts/sph.html)

# Spherical Harmonics

Any function that associates some quantity/value with a direction in 3D space can be thought of as a function defined on the domain of a `unit sphere`. Indeed, a "direction" is a unit vector, and the endpoint of that vector is always some point on the surface of a unit sphere centered at the origin.  

It just so happens that any continuous function defined on a sphere can be represented as an infinite weighted sum of some special polynomials. Those polynomials are, in fact, what we call "spherical harmonics". By truncating that sum and making it finite, we can approximate the function in question.  

Being able to approximate a potentially complex lighting environment by just summing up some polynomials is a very appealing proposition. But things other than lighting can be expressed as functions defined on a sphere too.

# Spherical Harmonics Functions

Spherical harmonic functions (or just “spherical harmonics” for short) are an infinite set of special functions defined on the surface of a sphere that form an orthonormal basis for the entire space of continuous functions defined on a sphere.  

It means that a continuous function defined on the sphere, no matter how complicated or difficult to evaluate, can be expressed as an infinite weighted sum of spherical harmonic functions.

# Spherical Harmonic Degree and Order

SH functions are split into numbered groups often called `“frequency bands”`. Each band has a number $l \in 0,\ 1,\ 2, ...$  associated with it. This number is called the "degree" of the functions within the band. The band with degree $l$ contains $2l+1$  functions. By convention, the functions within a band of degree $l$ are indexed from $-l$ to $l$, and that index is called the “order” of the function.

A spherical harmonic function with degree $l$ and order $m$ is usually denoted like
$$Y_l^m$$

In [1]:
import math
import numpy as np
import random

In [2]:
RECIP_PI = 1/math.pi
C = [
math.sqrt(RECIP_PI) * 0.5,
math.sqrt(3 * RECIP_PI) * 0.5,
math.sqrt(15 * RECIP_PI) * 0.5,
math.sqrt(5 * RECIP_PI) * 0.25,
math.sqrt(15 * RECIP_PI) * 0.25,
math.sqrt(70 * RECIP_PI) * 0.125,
math.sqrt(105 * RECIP_PI) * 0.5,
math.sqrt(42 * RECIP_PI) * 0.125,
math.sqrt(7 * RECIP_PI) * 0.25,
math.sqrt(105 * RECIP_PI) * 0.25
]
print(C)

[0.28209479177387814, 0.4886025119029199, 1.0925484305920792, 0.31539156525252005, 0.5462742152960396, 0.5900435899266435, 2.890611442640554, 0.4570457994644658, 0.3731763325901154, 1.445305721320277]


In [3]:
# SH basis functions up to degree l=3. 
# Source for SH basis function definitions:
# "Stupid Spherical Harmonics Tricks", Peter-Pike Sloan, 2008
def y00(x,y,z): return C[0] 
def y_11(x,y,z): return C[1] * y
def y01(x,y,z): return C[1] * z 
def y11(x,y,z): return C[1] * x
def y_22(x,y,z): return C[2] * y * x
def y_12(x,y,z): return C[2] * y * z
def y02(x,y,z): return C[3] * (3 * z * z - 1.0)
def y12(x,y,z): return C[2] * x * z
def y22(x,y,z): return C[4] * (x*x - y*y)
def y_33(x,y,z): return C[5] * y * (3*x*x - y*y)
def y_23(x,y,z): return C[6] * z * (y*x)
def y_13(x,y,z): return C[7] * y * (5*z*z -1)
def y03(x,y,z): return C[8] * z * (5*z*z - 3)
def y13(x,y,z): return C[7] * x * (5 * z * z - 1)
def y23(x,y,z): return C[9] * z * (x*x - y*y)
def y33(x,y,z): return C[5] * x * (x*x - 3*y*y)

In [4]:
# Evaluates SH basis functions with degrees up to and including l for the
# given direction d, returning the result as a Float32 array where each
# element is the value of the corresponding basis function.
# Only supports values of l <= 3.
def evalSHBasis(d, l):
  x = d[0]
  y = d[1]
  z = d[2]
  if l == 0:
      return np.array([y00(x, y, z)], dtype=np.float32)
  elif l == 1:
      return np.array([
        y00(x, y, z),  # l = 0
        y_11(x, y, z), # l = 1
        y01(x, y, z),
        y11(x, y, z)
    ], dtype=np.float32)
  elif l == 2:
      return np.array([
        y00(x, y, z),  # l = 0
        y_11(x, y, z), # l = 1
        y01(x, y, z),
        y11(x, y, z),
        y_22(x, y, z), # l = 2
        y_12(x, y, z),
        y02(x, y, z),
        y12(x, y, z),
        y22(x, y, z)
    ], dtype=np.float32)
  else:
      return np.array([
        y00(x, y, z),  # l = 0
        y_11(x, y, z), # l = 1
        y01(x, y, z),
        y11(x, y, z),
        y_22(x, y, z), # l = 2
        y_12(x, y, z),
        y02(x, y, z),
        y12(x, y, z),
        y22(x, y, z),
        y_33(x, y, z), # l = 3
        y_23(x, y, z),
        y_13(x, y, z),
        y03(x, y, z),
        y13(x, y, z),
        y23(x, y, z),
        y33(x, y, z)
    ], dtype=np.float32)

In [6]:
# SH 계수들에 스칼라 값을 곱하기 위한 도우미 함수
def mul_scalar_by_sh_coeffs(scalar, coeffs):
    # numpy 배열을 사용하면 for문이나 map 없이 스칼라 곱셈이 바로 적용됩니다.
    return scalar * coeffs

# 두 SH 계수들의 묶음을 더하기 위한 도우미 함수
def add_sh_coeffs(coeffs0, coeffs1):
    if len(coeffs0) == 0:
        return np.copy(coeffs1)
    # 배열 간의 덧셈도 직관적으로 바로 처리됩니다.
    return coeffs0 + coeffs1

# 기저 함수들이 정규 직교임을 검증하기 위해 간단한 몬테-카를로 적분을 사용
def test_basis_functions():
    num_samples = 100000
    num_basis_funcs = 16 # l=3까지 지원하므로 총 16개의 함수
    
    # inner_products[i]는 i-번째 SH 기저 함수를 자기자신을 포함한,
    # 다른 SH 기저 함수와 내적한 결과를 가지고 있음.
    # 초기값 0으로 채워진 크기 16의 numpy 배열 16개를 리스트로 생성합니다.
    inner_products = [np.zeros(num_basis_funcs, dtype=np.float32) for _ in range(num_basis_funcs)]
    
    s = 0
    while s < num_samples:
        # 구 상의 랜덤한 점을 생성하기 위해 간단한 기각 샘플링을 사용함:
        # * [-1, -1, -1] - [1, 1, 1] 큐브 내에서의 점을 생성하고
        # * 만약 점이 단위 구(원) 내부에 있다면, 정규화하고 해당 점을 사용함;
        # * 그렇지 않다면, 다시 시도함.
        xr = random.uniform(-1.0, 1.0)
        yr = random.uniform(-1.0, 1.0)
        zr = random.uniform(-1.0, 1.0)
        n = math.sqrt(xr*xr + yr*yr + zr*zr)
        
        if n > 1.0:
            continue
            
        s += 1
        d = np.array([xr/n, yr/n, zr/n], dtype=np.float32)
        
        # 생성된 샘플 지점에서의 SH 기저 함수를 계산하고
        # (외부에 eval_sh_basis 함수가 정의되어 있다고 가정합니다)
        basis_function_values = evalSHBasis(d, 3)
        
        # 부분 내적을 계산한 다음 총합에 더해줌.
        for b in range(num_basis_funcs):
            bv = basis_function_values[b]
            # numpy 배열에 스칼라 bv를 곱하여 배열 전체를 갱신합니다.
            partial_inner_products = basis_function_values * bv
            inner_products[b] = add_sh_coeffs(inner_products[b], partial_inner_products)
            
    # 몬테-카를로 적분의 마지막 단계: 적분 영역의 크기를 곱해주고 (단위 구의 경우 4pi)
    # 샘플의 수로 나눠줌.
    for b in range(num_basis_funcs):
        inner_products[b] = mul_scalar_by_sh_coeffs(4 * math.pi / num_samples, inner_products[b])
        
    # inner_products[i][j]에서 i == j인 경우엔 1에 아주 가까워야 하며,
    # i != j인 경우(서로 다른 기저 함수)엔 0에 아주 가까워야 합니다. (정규 직교성)
    for row in inner_products:
        print(row)

# 함수 실행을 위한 호출 (evalSHBasis가 구현되어 있을 때 정상 작동합니다)
test_basis_functions()

[ 1.0001496e+00 -1.6385148e-03 -2.9394641e-03  1.0751907e-03
 -8.8469344e-05  1.0023352e-03  2.2513913e-04 -4.1862824e-03
 -2.7463050e-03 -1.3130152e-04  8.8708138e-04  1.3219060e-03
  3.3162250e-03 -9.1602257e-04  1.5710525e-03 -3.3352489e-03]
[-1.6385148e-03  1.0020131e+00  7.7641709e-04 -6.8520021e-05
  3.6966256e-03 -4.9870485e-03  1.6807151e-03  5.8072549e-04
  1.4375468e-03  8.2291837e-05 -4.4452930e-03  5.5311313e-03
 -3.2295451e-05 -1.2945952e-03 -2.9977034e-03  2.0528804e-03]
[-2.9394641e-03  7.7641709e-04  1.0001881e+00 -3.2427004e-03
  5.8072503e-04 -1.7457503e-04  2.8353350e-04  7.4301235e-05
  1.0284921e-03 -2.0623682e-03 -1.7186328e-03  1.4129936e-03
  1.6418075e-03 -5.2634892e-03 -1.0427770e-02  1.8558643e-03]
[ 1.0751907e-03 -6.8520021e-05 -3.2427004e-03  9.9776441e-01
 -1.6481385e-03  5.8072462e-04 -1.1377450e-03 -2.9300218e-03
 -1.6516428e-03  2.5890102e-03 -2.0540394e-03 -1.2945961e-03
  6.5563136e-04 -6.9768135e-03  1.0065645e-04 -9.6305466e-04]
[-8.8469344e-05  3.6

# Spherical Harmonic Definition

$$Y_l^m(\theta,\ \phi) = (-1)^m \sqrt{\frac{(2l+1)}{4\pi}\frac{(l-m)!}{(l+m)!}}P_l^m(cos\theta)e^{im\phi}$$